In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Sellers

🎯 Our goal is to find sellers who have repeatedly been underperforming vs. others, and understand why.  
This will help us shape our recommendations about how to improve Olist's profit margin for the future.

❗️ Long Notebook. Once you've read a section, you can collapse it.
<details>
    <summary> <i>[Reminder] Notebook best practices</i></summary>

- Code your logic so that your Notebook can always be run from top to bottom without crashing (`Cell --> Run All`)
- Name your variables carefully 
- Use dummy names such as `tmp` for intermediary steps when you know you won't need them later
- Clear your code and merge cells when relevant to minimize Notebook size (`Shift-M`)
- Hide your cell output if you don't need to see it anymore (double click on the red `Out[]:` section to the left of your cell).
- Make heavy use of jupyter nbextention `Collapsable Headings` and `Table of Content` (call a TA if you can't find them)
- Use the following shortcuts 
    - `a` to insert a cell above
    - `b` to insert a cell below
    - `dd` to delete a cell
    - `esc` and `arrows` to move between cells
    - `Shift-Enter` to execute cell and move focus to the next one
    - use `Shift + Tab` when you're between method brackets e.g. `groupby()` to get the docs! Repeat a few times to open it permanently

</details>





## 1 - `olist/seller.py`  

In a process similar to `order.py`, we have coded for you the module `olist/seller.py` containing a class `Seller` with a method `Seller().get_training_data` that will return a DataFrame with the following features:
  
| feature_name 	| type 	| description 	|
|:---	|:---:	|:---	|
| `seller_id` 	| str 	| the id of the seller **UNIQUE** 	|
| `seller_city` 	| str 	| the city where seller is located 	|
| `seller_state` 	| str 	| the state where seller is located 	|
| `delay_to_carrier` 	| float 	| returns 0 if the order is delivered before the shipping_limit_date, otherwise the value of the delay 	|
| `wait_time` 	| float 	| average wait_time (duration of deliveries) per seller 	|
| `date_first_sale` 	| datetime 	| date of the first sale on Olist 	|
| `date_last_sale` 	| datetime 	| date of the last sale on Olist 	|
| `months_on_olist` 	| float 	| round number of months  on Olist	|
| `share_of_five_stars` 	| float 	| share of five-star reviews for orders in which the seller was involved 	|
| `share_of_one_stars` 	| float 	| share of one-star reviews for orders in which the seller was involved 	|
| `review_score` 	| float 	| average review score for orders in which the seller was involved 	|
| `n_orders` 	| int 	| number of unique orders the seller was involved with 	|
| `quantity` 	| int 	| total number of items sold by this seller 	|
| `quantity_per_order` 	| float 	| average number of items per order for this seller 	|
| `sales` 	| float 	| total sales associated with this seller (excluding freight value) in BRL 	|  

❓ **Import your new class below and check out your training dataframe !** Take time to look at the code and understand exactly what has been computed for you

In [4]:
# YOUR CODE HERE
from olist.seller import Seller
from olist.order import Order
sellers = Seller().get_training_data()
orders = Order().get_training_data()

/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/seller.py:100: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  df = orders_sellers.groupby('seller_id').agg({
/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/seller.py:100: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df = orders_sellers.groupby('seller_id').agg({
/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/order.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_i

🤔 One last thing remains to be computed for each seller:
* the proportion of extremely high reviews (`share_of_five_stars`) and the proportion of extremely poor reviews (`share_of_one_stars`)
* the (average) `review_score`

😱 Each low-rated order will indeed have a negative impact on Olist's reputation and this is modeled by the `cost_of_review`.  

This will help us compute the total `cost_of_review` per seller later on!

❓ **Implement the last method that has been left for you `get_review_score()`**

In [ ]:
# YOUR CODE HERE
from olist.data import Olist
olist = Olist()
data = olist.get_data()
order_items = data['order_items'].copy()
sellers_order = sellers.merge(order_items, on = 'seller_id')
sellers_order = sellers_order.merge(orders, on = 'order_id')

seller_five = sellers_order.groupby('seller_id')['dim_is_five_star'].sum()
sum_seller_five = seller_five.sum() 

share_of_five_stars = seller_five/sum_seller_five
share_of_five_stars = share_of_five_stars.to_frame().reset_index()
share_of_five_stars

In [70]:
share_of_five_stars


seller_id
0015a82c2db000af6aaaf3ae2ecb0532    3.666667
001cca7ae9ae17fb1caed9dfb1094831    3.965368
002100f778ceb8431b7a1020ff7ab48f    4.036364
003554e2dce176b5555353e4f3555ac8    5.000000
004c9cd9d87a3c30c522c48c4fc07416    4.145349
                                      ...   
ffc470761de7d0232558ba5e786e57b7    4.300000
ffdd9f82b9a447f6f8d4b91554cc7dd3    4.250000
ffeee66ac5d5a62fe688b9d26f83f534    4.214286
fffd5413c0700ac820c7069d66d98c89    3.912281
ffff564a4f9085cd26170f4732393726    3.250000
Name: review_score, Length: 2965, dtype: float64

In [87]:
seller_one = sellers_order.groupby('seller_id')['dim_is_one_star'].sum()
sum_seller_one = seller_one.sum()
share_of_one_stars = seller_one/sum_seller_one
share_of_one_stars = share_of_one_stars.to_frame().reset_index()
share_of_one_stars= share_of_one_stars.rename(columns={'dim_is_one_star': 'share_of_one_stars'})
share_of_one_stars

,seller_id,share_of_one_stars
0,0015a82c2db000af6aaaf3ae2ecb0532,0.000080
1,001cca7ae9ae17fb1caed9dfb1094831,0.002306
2,002100f778ceb8431b7a1020ff7ab48f,0.000477
3,003554e2dce176b5555353e4f3555ac8,0.000000
4,004c9cd9d87a3c30c522c48c4fc07416,0.001113
...,...,...
2960,ffc470761de7d0232558ba5e786e57b7,0.000159
2961,ffdd9f82b9a447f6f8d4b91554cc7dd3,0.000080
2962,ffeee66ac5d5a62fe688b9d26f83f534,0.000159
2963,fffd5413c0700ac820c7069d66d98c89,0.000795


In [94]:
avg_score = sellers_order.groupby('seller_id')['review_score'].mean()\
    .to_frame().reset_index()
avg_score['review_score'] = avg_score['review_score'].round(1)
sellers_df = share_of_one_stars.merge(share_of_five_stars)\
    .merge(avg_score, on = 'seller_id')
sellers_df

,seller_id,share_of_one_stars,dim_is_five_star,review_score
0,0015a82c2db000af6aaaf3ae2ecb0532,0.000080,0.000032,3.7
1,001cca7ae9ae17fb1caed9dfb1094831,0.002306,0.001911,4.0
2,002100f778ceb8431b7a1020ff7ab48f,0.000477,0.000490,4.0
3,003554e2dce176b5555353e4f3555ac8,0.000000,0.000016,5.0
4,004c9cd9d87a3c30c522c48c4fc07416,0.001113,0.001643,4.1
...,...,...,...,...
2960,ffc470761de7d0232558ba5e786e57b7,0.000159,0.000316,4.3
2961,ffdd9f82b9a447f6f8d4b91554cc7dd3,0.000080,0.000205,4.2
2962,ffeee66ac5d5a62fe688b9d26f83f534,0.000159,0.000142,4.2
2963,fffd5413c0700ac820c7069d66d98c89,0.000795,0.000505,3.9


In [100]:
order_items = data['order_items']
sellers = order_items.merge(orders, on = 'order_id')

seller_five = sellers.groupby('seller_id')['dim_is_five_star'].sum()
sum_seller_five = seller_five.sum()
share_of_five_stars = seller_five/sum_seller_five
share_of_five_stars = share_of_five_stars.to_frame().reset_index()

seller_one = sellers.groupby('seller_id')['dim_is_one_star'].sum()
sum_seller_one = seller_one.sum()
share_of_one_stars = seller_one/sum_seller_one
share_of_one_stars = share_of_one_stars.to_frame().reset_index()

avg_score = sellers.groupby('seller_id')['review_score'].mean()\
    .to_frame().reset_index()
sellers_df = share_of_one_stars.merge(share_of_five_stars)\
    .merge(avg_score, on = 'seller_id')
sellers_df

,seller_id,dim_is_one_star,dim_is_five_star,review_score
0,0015a82c2db000af6aaaf3ae2ecb0532,0.000080,0.000032,3.666667
1,001cca7ae9ae17fb1caed9dfb1094831,0.002306,0.001911,3.965368
2,002100f778ceb8431b7a1020ff7ab48f,0.000477,0.000490,4.036364
3,003554e2dce176b5555353e4f3555ac8,0.000000,0.000016,5.000000
4,004c9cd9d87a3c30c522c48c4fc07416,0.001113,0.001643,4.145349
...,...,...,...,...
2960,ffc470761de7d0232558ba5e786e57b7,0.000159,0.000316,4.300000
2961,ffdd9f82b9a447f6f8d4b91554cc7dd3,0.000080,0.000205,4.250000
2962,ffeee66ac5d5a62fe688b9d26f83f534,0.000159,0.000142,4.214286
2963,fffd5413c0700ac820c7069d66d98c89,0.000795,0.000505,3.912281


🧪 Test your code below

In [101]:
from nbresult import ChallengeResult

tmp = Seller().get_training_data()
result = ChallengeResult('seller',
    shape = tmp.shape,
    median = tmp.review_score.median(),
    columns = tmp.columns
)
result.write()
print(result.check())

/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/seller.py:100: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  df = orders_sellers.groupby('seller_id').agg({
/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/seller.py:100: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df = orders_sellers.groupby('seller_id').agg({
/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/order.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_i


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/lanwy/.pyenv/versions/3.12.9/envs/lewagon/bin/python
cachedir: .pytest_cache
rootdir: /Users/lanwy/code/lanwyb/04-Decision-Science/03-Linear-Regression/data-sellers/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 3 items

test_seller.py::TestSeller::test_column_names PASSED                     [ 33%]
test_seller.py::TestSeller::test_median_review_score PASSED              [ 66%]
test_seller.py::TestSeller::test_shape FAILED                            [100%]

=================================== FAILURES ===================================
____________________________ TestSeller.test_shape _____________________________

self = <tests.test_seller.TestSeller testMethod=test_shape>

    def test_shape(self):
>       self.assertEqual(self.result.shape, (2967, 15),
                         msg="Expected exactly 2967 rows 

💡 **Not getting the exact number of rows?**
<details><summary>Do you have an extra 3 rows?</summary>
Did you do a left or right join? We see why, but here we are only interested in sellers who actually received reviews, and we took an inner join.
</details>
<details><summary>Are you missing 2 rows?</summary>
Did you use <code>Orders().get_training_data()</code>? That's a valid option, but it's a bit overkill if we're only interested in reviews, no? Remember how that method does a lot of calculations. And the number of columns it returns: we don't need most of them. Find another method in the <code>Order</code> class that would be better tailored to what we need.
</details>

## 2 - Sellers' Exploration

### (2.1) Plots

👉 Let's start with some initial ***`EDA - Exploratory Data Analysis`*** about these sellers.

- 📈 Plot the distribution of each numerical variable of the dataset in one large figure
- 👀 Do you notice any outliers?
- What's the median of orders per seller ❓
- How does the distribution of this variable look like ❓

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

💡There seems to be a group of sellers which stands out for having very low review scores! 

📊 Let's investigate graphically it:
* Using `plotly`, create a `scatterplot` of `delay_to_carrier` against `wait_time`, varying bubble size by total `sales` for that seller, and coloring by `review_score`. 

In [ ]:
# YOUR CODE HERE

Feel free to change values `x`, `y`, `color` and `size` to try identify who are the worst sellers

### (2.2) Model out `review_score` with OLS

⚠️ Scatter plots have their limits. 

💡 A more rigorous way to explain **`sellers' review_score`** is to **model the impact of various features on `review_score` with a `multivariate-OLS` in `statsmodels`**.

👉 Create an OLS with numerical features of your choice. 

❓ What are the most impactful ones? 

⚠️ Don't forget to standardize your features using the `standardize`function below to compare the regression coefficients together. 

In [ ]:
def standardize(df, features):
    df_standardized = df.copy()
    for f in features:
        mu = df[f].mean()
        sigma = df[f].std()
        df_standardized[f] = df[f].map(lambda x: (x - mu) / sigma)
    return df_standardized

In [ ]:
# YOUR CODE HERE

📊 Draw a `bar_plot` with sorted coefficients.

In [ ]:
# YOUR CODE HERE

👉 Finally, investigate your model's performance (`R-squared`) and `residuals`

In [ ]:
# YOUR CODE HERE

👉 Compare the real review scores and the predicted scores by showing them on the same graph.

In [ ]:
# YOUR CODE HERE

👉 Plot the residuals

In [ ]:
# YOUR CODE HERE

### (2.3) Add the `seller_state` to your analysis

❓ We haven't used information about `seller_state` yet.  
- Create a new OLS model regressing `review_score` on only on `seller_states` .
- Analyse your significant features using `return_significative_coef(model)` coded for you in `olist/utils.py`
- What are the best states in terms of `review_score`? 

<details>
    <summary>- Hints -</summary>
        
⚠️ Be careful, `seller_state` is a categorical feature. 
    
💡 Use `C(a_cat_feature)` in the formula to tell the linear regression model which variables are categorical variables. It will create one boolean variable `is_cat_feature_xx` **per unique category** 

</details>

In [ ]:
# YOUR CODE HERE

☝️ Some states indeed have _significantly_ better reviews than others on average. 

🤔 Is it due to some lower `quantity_per_order`, lower `wait_time`, or `delay_to_carrier`?  Or is it due to some other factors that we haven't collected data about?

❓ **Try to isolate the impact of the `seller_state` from the rest by adding other continuous features to your OLS until `seller_states` is no longer statistically siginificant!**

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

☝️ After adding `wait_time` to our analysis, none of the 22 dummy variables `is_seller_state_xx` are statistically significant:

Given our small dataset (most states have a very limited number of sellers):
- We _cannot conclude_ that "some states are inherently better than other for reasons that would be independent of the `wait_time`" 
- In other words, we _cannot reject the hypothesis_ that "seller_state has no impact on review_score, other than through `wait_time`"

🏁 Congratulations!

💾 Commit and push :
* your ` sellers.ipynb`notebook 
* as well as `seller.py`